In [74]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape, Polygon
import numpy as np
from pathlib import Path
import os
import sys
from itertools import product

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
PARENT_DIRECTORY = os.path.dirname(CURRENT_DIRECTORY)
print(PARENT_DIRECTORY)

BASE_DATASET_PATH = Path(PARENT_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [67]:
def combine_multiple_df(start_year: int, end_year: int, filename: str):
    all_dfs = []
    year = start_year

    if filename == "grid_classification":
        for year in range(start_year, end_year + 1):
            year_str = str(year)
            filepath = Path(BASE_DATASET_PATH / f"geospatial_data/{year_str}_geospatial/{filename}.csv")

            if filepath.exists():
                temp_df = pd.read_csv(filepath)
                all_dfs.append(temp_df)
            else:
                print(f"Warning: File not found at {filepath}")

    else: 
        for year in range(start_year, end_year + 1):
            year_str = str(year)
            filepath = Path(BASE_DATASET_PATH / f"geospatial_data/{year_str}_geospatial/{filename}_{year_str}.csv")

            if filepath.exists():
                temp_df = pd.read_csv(filepath)
                all_dfs.append(temp_df)
            else:
                print(f"Warning: File not found at {filepath}")

    if all_dfs:
        output_df = pd.concat(all_dfs, ignore_index = True)
    else:
        output_df = pd.DataFrame()
        
    print("Done")
    return output_df

In [ ]:
output_df = combine_multiple_df(2014, 2020)
output_df = output_df.sort_values(by = ["date", "time"]).copy()

Done


In [ ]:
output_df.head()

,grid_id,grid_row,grid_col,id,date,time,location_type,location_type_other,hectare_classification
1892,134367,118,388,5455,2014/01/01,04:47:03,Home Residence,Level 10,NaN
1949,140229,98,405,5454,2014/01/01,06:23:30,Industrial Place,level 1,workforce
359,61061,164,176,5456,2014/01/01,09:29:55,Home Residence,level 15,major_road
1470,108054,101,312,5459,2014/01/01,11:01:39,Home Residence,level 7,major_road
23,32322,143,93,5458,2014/01/01,11:15:35,Home Residence,level 4,NaN


In [ ]:
output_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18744 entries, 1892 to 18295
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   grid_id                 18744 non-null  int64 
 1   grid_row                18744 non-null  int64 
 2   grid_col                18744 non-null  int64 
 3   id                      18744 non-null  int64 
 4   date                    18744 non-null  object
 5   time                    18689 non-null  object
 6   location_type           18744 non-null  object
 7   location_type_other     6634 non-null   object
 8   hectare_classification  13091 non-null  object
dtypes: int64(4), object(5)
memory usage: 1.4+ MB


In [ ]:
output_filepath = Path(BASE_DATASET_PATH / "4_OHCA_WITH_GRID_CLASSIFICATION.csv")
output_df.to_csv(output_filepath)

In [57]:
output_filepath = Path(BASE_DATASET_PATH / "4_OHCA_WITH_GRID_CLASSIFICATION.csv")
combined_df = pd.read_csv(output_filepath)
combined_df.head()

,Unnamed: 0,grid_id,grid_row,grid_col,id,date,time,location_type,location_type_other,hectare_classification
0,1892,134367,118,388,5455,2014/01/01,04:47:03,Home Residence,Level 10,NaN
1,1949,140229,98,405,5454,2014/01/01,06:23:30,Industrial Place,level 1,workforce
2,359,61061,164,176,5456,2014/01/01,09:29:55,Home Residence,level 15,major_road
3,1470,108054,101,312,5459,2014/01/01,11:01:39,Home Residence,level 7,major_road
4,23,32322,143,93,5458,2014/01/01,11:15:35,Home Residence,level 4,NaN


### Filling in hectare_classification with the location_type from Paros data
As I was not able to classify each of the hectare fully, the NaN values of the hectare_classification will be filled depending on the location_type column stated by the Paros dataset

I am trying to be careful to not map too other types such as Street/Highway or Place of Recreation as it might not accurate map to the classification I had.
- Home Residence -> residential
- Industrial Place -> workforce
- Nursing Home -> residential


Question: 
- How to classify Healthcare Facility (location_type)?


#### Following the location_type from PAROS dataset filled up most of the null data.

In [58]:
mapping = {
    'Home Residence': 'residential',
    'Industrial Place': 'workforce',
    'Nursing Home': 'residential'
}

# Identify rows where 'hectare_classification' is NaN
is_nan_hectare = combined_df['hectare_classification'].isnull()

# Use the mapping to generate replacement values ONLY for the NaN rows
replacement_values = combined_df.loc[is_nan_hectare, 'location_type'].map(mapping)

# Use .fillna() to update the 'hectare_classification' column
# It only fills NaN values with the corresponding values from the 'replacement_values' Series
combined_df['hectare_classification'] = combined_df['hectare_classification'].fillna(replacement_values)

print("Updated DataFrame (first 5 rows that had NaN in hectare_classification):")
# combined_df.loc[is_nan_hectare].head()
combined_df.info()

Updated DataFrame (first 5 rows that had NaN in hectare_classification):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18744 entries, 0 to 18743
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Unnamed: 0              18744 non-null  int64 
 1   grid_id                 18744 non-null  int64 
 2   grid_row                18744 non-null  int64 
 3   grid_col                18744 non-null  int64 
 4   id                      18744 non-null  int64 
 5   date                    18744 non-null  object
 6   time                    18689 non-null  object
 7   location_type           18744 non-null  object
 8   location_type_other     6634 non-null   object
 9   hectare_classification  17681 non-null  object
dtypes: int64(5), object(5)
memory usage: 1.4+ MB


In [59]:
# fill NaN of the hectare_classification column with Unclassified
combined_df['hectare_classification'] = combined_df['hectare_classification'].fillna('Unclassified')
combined_df["month"] = pd.to_datetime(combined_df["date"]).dt.month
combined_df["hectare_classification"].unique()

array(['residential', 'workforce', 'major_road', 'mall', 'school',
       'Unclassified', 'airport', 'street_shops'], dtype=object)

In [ ]:
# --- STEP 1: Perform One-Hot Encoding ---
# 'drop_first=True' is essential for regression to avoid multicollinearity (N-1 features).
# The category dropped (first alphabetically, which is 'airport' in this case) 
# becomes the reference group.
hectare_dummies = pd.get_dummies(
    combined_df['hectare_classification'], 
    prefix='class', 
    drop_first=True # the Unclassified category of the hectare_classification column is dropped. so if all of the other classification are False, it would mean that hectare is Unclassified
)

# --- STEP 2: Concatenate the new dummy columns with the original DataFrame ---
combined_df = pd.concat([combined_df, hectare_dummies], axis=1)

# --- STEP 3: Drop the original (now redundant) categorical column ---
# combined_df = combined_df.drop('hectare_classification', axis=1)

# --- Verification ---
print("--- DataFrame Head after One-Hot Encoding (Should show 8 new class columns) ---")
combined_df.head()

--- DataFrame Head after One-Hot Encoding (Should show 8 new class columns) ---


,Unnamed: 0,grid_id,grid_row,grid_col,id,date,time,location_type,location_type_other,hectare_classification,month,class_airport,class_major_road,class_mall,class_residential,class_school,class_street_shops,class_workforce
0,1892,134367,118,388,5455,2014/01/01,04:47:03,Home Residence,Level 10,residential,1,False,False,False,True,False,False,False
1,1949,140229,98,405,5454,2014/01/01,06:23:30,Industrial Place,level 1,workforce,1,False,False,False,False,False,False,True
2,359,61061,164,176,5456,2014/01/01,09:29:55,Home Residence,level 15,major_road,1,False,True,False,False,False,False,False
3,1470,108054,101,312,5459,2014/01/01,11:01:39,Home Residence,level 7,major_road,1,False,True,False,False,False,False,False
4,23,32322,143,93,5458,2014/01/01,11:15:35,Home Residence,level 4,residential,1,False,False,False,True,False,False,False


In [71]:
filepath = Path(BASE_DATASET_PATH / "geospatial_data/hectare_grid.gpkg")
hectare_grid_df = gpd.read_file(filepath)

In [72]:
hectare_grid_df = hectare_grid_df.drop("geometry", axis = 1)
hectare_grid_df

,id,left,top,right,bottom,row_index,col_index
0,1,2666.7735,50257.9506,2766.7735,50157.9506,0,0
1,2,2666.7735,50157.9506,2766.7735,50057.9506,1,0
2,3,2666.7735,50057.9506,2766.7735,49957.9506,2,0
3,4,2666.7735,49957.9506,2766.7735,49857.9506,3,0
4,5,2666.7735,49857.9506,2766.7735,49757.9506,4,0
...,...,...,...,...,...,...,...
186143,186144,56366.7735,16157.9506,56466.7735,16057.9506,341,537
186144,186145,56366.7735,16057.9506,56466.7735,15957.9506,342,537
186145,186146,56366.7735,15957.9506,56466.7735,15857.9506,343,537
186146,186147,56366.7735,15857.9506,56466.7735,15757.9506,344,537


In [75]:
MAX_GRID_ID = hectare_grid_df["id"].max()
grid_ids = np.arange(1, MAX_GRID_ID + 1)

# The range of years (2014 to 2020 inclusive)
years = np.arange(2014, 2021)

# The range of weeks (1 to 52 inclusive)
weeks = np.arange(1, 53)

# Create a list of tuples for all combinations
index_combinations = list(product(grid_ids, years, weeks))

# Convert the list of tuples into the target DataFrame
grid_time_index_df = pd.DataFrame(
    index_combinations, 
    columns=['grid_id', 'year', 'week_of_year']
)
# --- 4. Verification and Summary ---
total_rows = len(grid_time_index_df)
print(f"Total rows generated: {total_rows:,}")
print(f"Expected rows: {MAX_GRID_ID * len(years) * len(weeks):,}")

# Display the head and tail to show the structure and range
print("\n--- Constructed Grid-Time Index DataFrame Head ---")
print(grid_time_index_df.head())

print("\n--- Constructed Grid-Time Index DataFrame Tail ---")
print(grid_time_index_df.tail())

Total rows generated: 67,757,872
Expected rows: 67,757,872

--- Constructed Grid-Time Index DataFrame Head ---
   grid_id  year  week_of_year
0        1  2014             1
1        1  2014             2
2        1  2014             3
3        1  2014             4
4        1  2014             5

--- Constructed Grid-Time Index DataFrame Tail ---
          grid_id  year  week_of_year
67757867   186148  2020            48
67757868   186148  2020            49
67757869   186148  2020            50
67757870   186148  2020            51
67757871   186148  2020            52


In [76]:
filepath = Path(BASE_DATASET_PATH / "4_checkpoint_grid_by_weeks.csv")
grid_time_index_df.to_csv(filepath)